# Measured Intrinsic Wavefront (MIW): OCS / CCS map reader

**Author:** Aaron Roodman &nbsp;|&nbsp; **Purpose:** read and plot the telescope-fixed (OCS) and
camera-fixed (CCS) Measured Intrinsic Wavefront maps for the Rubin AOS.

This is a standalone reader for the `intrinsic_split_maps.parquet` product. It needs only
`numpy`, `matplotlib`, `astropy`, and (for the optional lookup helper) `scipy` — **no LSST stack**
— so it can be used directly in the AOS online system.

### What the file contains
`intrinsic_split_maps.parquet` is an astropy Table on a regular focal-plane disk grid:

| column | meaning |
|---|---|
| `thx_deg`, `thy_deg` | field angle (degrees), OCS frame |
| `Z{j}_OCS` | telescope-fixed MIW for Noll Zernike *j* (µm) |
| `Z{j}_CCS` | camera-fixed MIW for Noll Zernike *j* (µm) |

Each measured map at rotator angle θ is modelled as `Z(r,φ) = O(r,φ) + C(r, φ - s·θ)`:
`O` is the telescope-fixed (OCS) part, `C` is the camera-fixed (CCS) part that rotates with the
Camera Rotator. For Zernikes fit OCS-only, `Z{j}_CCS` is identically zero. The table `.meta`
records the rotator angles used (`thetas_deg`), any `rotator_select` subset, the `ocs_only`
Zernikes, the rotation sign `s`, and `build_from`.

## 1. Parameters

In [ ]:
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

# Versioned OCS/CCS maps from the calibration store (see aos/calibration/).
version = "v1"
maps_path = f"calibration/miw/{version}/intrinsic_split_maps.parquet"
# For online/AOS use, set maps_path to an absolute path to the deployed file.

zernikes = "all"        # "all", or a list of Noll indices e.g. [4, 5, 6, 7, 8]
fp_radius_deg = 1.75    # focal-plane radius for plot limits (deg)
pct = 98                # color scale percentile (per Zernike)

## 2. Read the maps

In [ ]:
t = Table.read(maps_path, format="parquet")
thx = np.asarray(t["thx_deg"], dtype=float)
thy = np.asarray(t["thy_deg"], dtype=float)

# discover the Zernikes present from the Z{j}_OCS columns
js = sorted({int(m.group(1)) for c in t.colnames
             for m in [re.match(r"Z(\d+)_OCS$", c)] if m})
if zernikes != "all":
    js = [j for j in js if j in zernikes]
ocs_only = set(int(j) for j in (t.meta.get("ocs_only") or []))

print(f"{len(thx)} field points;  Zernikes available: {js}")
print("metadata:")
for k, v in t.meta.items():
    print(f"  {k}: {v}")

## 3. Helper: focal-plane map

In [ ]:
def plot_map(ax, vals, title, vlim, cmap="RdBu_r"):
    """Filled-contour map of a per-field-point value on (thx, thy)."""
    v = np.asarray(vals, dtype=float)
    fin = np.isfinite(v)
    tcf = ax.tricontourf(thx[fin], thy[fin], v[fin],
                         levels=np.linspace(-vlim, vlim, 21), cmap=cmap, extend="both")
    ax.add_patch(plt.Circle((0, 0), fp_radius_deg, fill=False, ec="k", lw=0.6, alpha=0.4))
    ax.set_aspect("equal")
    ax.set_xlim(-fp_radius_deg, fp_radius_deg); ax.set_ylim(-fp_radius_deg, fp_radius_deg)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("thx [deg]"); ax.set_ylabel("thy [deg]")
    return tcf


def vlim_for(*arrays):
    """Symmetric color limit from the robust percentile over the given arrays."""
    vals = np.concatenate([np.asarray(a, float)[np.isfinite(a)] for a in arrays])
    return max(float(np.nanpercentile(np.abs(vals), pct)), 1e-6) if vals.size else 1.0

## 4. Plot the OCS and CCS maps
One row per Zernike: telescope-fixed (OCS) on the left, camera-fixed (CCS) on the right, shared color scale.

In [ ]:
for j in js:
    O = np.asarray(t[f"Z{j}_OCS"], dtype=float)
    C = np.asarray(t[f"Z{j}_CCS"], dtype=float)
    vlim = vlim_for(O, C)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.6), layout="constrained")
    tcf = plot_map(axs[0], O, f"Z{j}  OCS (telescope)", vlim)
    ccs_note = "   [OCS-only: C≈0]" if j in ocs_only else ""
    plot_map(axs[1], C, f"Z{j}  CCS (camera){ccs_note}", vlim)
    fig.colorbar(tcf, ax=axs, shrink=0.85, label="µm")
    fig.suptitle(f"Measured Intrinsic Wavefront — Z{j}", fontsize=12)
    plt.show()

## 5. Optional: evaluate the MIW at arbitrary field angles
For online use — look up the OCS or CCS Zernike value at any field position (e.g. a WFS donut).
Returns NaN outside the sampled field.

In [ ]:
from scipy.interpolate import LinearNDInterpolator

def miw_interpolator(j, frame="OCS"):
    """LinearNDInterpolator for Z{j}_{frame}: call f(thx_deg, thy_deg) -> value (µm)."""
    v = np.asarray(t[f"Z{j}_{frame}"], dtype=float)
    pts = np.column_stack([thx, thy])
    fin = np.isfinite(v)
    return LinearNDInterpolator(pts[fin], v[fin])

# example: Z5 telescope-fixed MIW at field angle (0.3, 1.2) deg
f_ocs = miw_interpolator(5, "OCS")
print("Z5 OCS at (0.3, 1.2) deg =", float(f_ocs(0.3, 1.2)), "um")